# Project 1: Business Sales Analysis
## Complete Retail Sales Analytics with Customer Segmentation

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

from utils.data_analysis_utils import utils
print("✅ All imports successful")


✅ All imports successful


In [ ]:
# ── Generate synthetic sales dataset ──────────────────────────
np.random.seed(42)
n_records = 15000
dates = pd.date_range('2023-01-01', '2024-12-31', freq='h')
order_dates = np.random.choice(dates, n_records)

products = {
    'Laptop':     {'category': 'Electronics',  'price_range': (800, 2500)},
    'Smartphone': {'category': 'Electronics',  'price_range': (500, 1500)},
    'Tablet':     {'category': 'Electronics',  'price_range': (300, 1000)},
    'Headphones': {'category': 'Accessories',  'price_range': ( 50,  300)},
    'Keyboard':   {'category': 'Accessories',  'price_range': ( 30,  150)},
    'Mouse':      {'category': 'Accessories',  'price_range': ( 20,  100)},
    'Monitor':    {'category': 'Electronics',  'price_range': (200,  800)},
    'Charger':    {'category': 'Accessories',  'price_range': ( 15,   60)},
    'Smartwatch': {'category': 'Electronics',  'price_range': (200,  600)},
    'Speaker':    {'category': 'Accessories',  'price_range': ( 50,  400)},
}

data = []
for i in range(n_records):
    product = np.random.choice(list(products.keys()))
    info    = products[product]
    price   = np.random.uniform(*info['price_range'])
    qty     = np.random.choice([1,2,3,4,5], p=[0.5,0.25,0.15,0.07,0.03])
    month   = month = order_dates[i].astype('M8[ms]').astype(datetime).month
    if month in [11, 12]:
        qty = int(qty * np.random.uniform(1.2, 1.8))
    elif month in [6, 7, 8]:
        qty = int(qty * np.random.uniform(0.9, 1.2))
    data.append({
        'OrderID':        f'ORD{str(i+1).zfill(8)}',
        'OrderDate':      order_dates[i],
        'CustomerID':     np.random.randint(1000, 3000),
        'Product':        product,
        'Category':       info['category'],
        'Quantity':       max(1, qty),
        'UnitPrice':      round(price, 2),
        'Region':         np.random.choice(['North','South','East','West','Central'],
                                           p=[0.25,0.2,0.2,0.25,0.1]),
        'PaymentMethod':  np.random.choice(['Credit Card','Debit Card','PayPal','Apple Pay','Google Pay'],
                                           p=[0.4,0.25,0.15,0.1,0.1]),
        'CustomerAge':    np.random.randint(18, 75),
        'CustomerGender': np.random.choice(['Male','Female','Other'], p=[0.48,0.48,0.04]),
        'DiscountPercent':np.random.choice([0,5,10,15,20,25], p=[0.5,0.2,0.15,0.08,0.05,0.02]),
    })

df_sales = pd.DataFrame(data)
df_sales['SalesAmount']     = df_sales['Quantity'] * df_sales['UnitPrice']
df_sales['DiscountedAmount']= df_sales['SalesAmount'] * (1 - df_sales['DiscountPercent']/100)
df_sales['Profit']          = df_sales['DiscountedAmount'] * np.random.uniform(0.2, 0.45, len(df_sales))

# add 300 missing ages
missing_idx = np.random.choice(df_sales.index, 300, replace=False)
df_sales.loc[missing_idx, 'CustomerAge'] = np.nan

print(f"✅ {len(df_sales):,} records generated")
print(f"💰 Total Revenue : ${df_sales['SalesAmount'].sum():,.2f}")
print(f"📊 Total Profit  : ${df_sales['Profit'].sum():,.2f}")


AttributeError: 'numpy.datetime64' object has no attribute 'month'

In [ ]:
# ── EDA ───────────────────────────────────────────────────────
report = utils.comprehensive_eda(df_sales)
print("Shape   :", report['basic_info']['shape'])
print("Missing Values:\n", report['missing_values'])
print("Duplicates:", report['duplicates'])


In [ ]:
# ── Clean ─────────────────────────────────────────────────────
df = utils.clean_dataframe(df_sales, handle_missing='median', drop_duplicates=True)
print(f"Cleaned shape: {df.shape}")


In [ ]:
# ── Time-series features ──────────────────────────────────────
df['OrderDate'] = pd.to_datetime(df['OrderDate'])
df['Year']      = df['OrderDate'].dt.year
df['Month']     = df['OrderDate'].dt.month
df['DayOfWeek'] = df['OrderDate'].dt.dayofweek
df['Hour']      = df['OrderDate'].dt.hour

monthly = df.groupby(['Year','Month']).agg(
    Revenue=('SalesAmount','sum'),
    Profit =('Profit','sum'),
    Orders =('OrderID','count')
).reset_index()
monthly['Label'] = (monthly['Year'].astype(str) + '-' +
                    monthly['Month'].map({1:'Jan',2:'Feb',3:'Mar',4:'Apr',
                                          5:'May',6:'Jun',7:'Jul',8:'Aug',
                                          9:'Sep',10:'Oct',11:'Nov',12:'Dec'}))

os.makedirs('visualizations', exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes[0,0].plot(monthly['Label'], monthly['Revenue']/1000, marker='o', color='#2E86AB')
axes[0,0].set_title('Monthly Revenue Trend', fontweight='bold')
axes[0,0].set_ylabel('Revenue ($K)')
axes[0,0].tick_params(axis='x', rotation=45)
axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(monthly['Label'], monthly['Profit']/1000, marker='s', color='#A23B72')
axes[0,1].set_title('Monthly Profit Trend', fontweight='bold')
axes[0,1].set_ylabel('Profit ($K)')
axes[0,1].tick_params(axis='x', rotation=45)
axes[0,1].grid(True, alpha=0.3)

hourly = df.groupby('Hour')['OrderID'].count()
axes[1,0].bar(hourly.index, hourly.values, color='#F18F01')
axes[1,0].set_title('Orders by Hour of Day', fontweight='bold')
axes[1,0].set_xlabel('Hour'); axes[1,0].set_ylabel('Orders')

dow = df.groupby('DayOfWeek')['OrderID'].count()
axes[1,1].bar(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'], dow.values, color='#C73E1D')
axes[1,1].set_title('Orders by Day of Week', fontweight='bold')

plt.tight_layout()
plt.savefig('visualizations/monthly_sales_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: visualizations/monthly_sales_trend.png")


In [ ]:
# ── Product performance ───────────────────────────────────────
prod = df.groupby('Product').agg(
    Revenue =('SalesAmount','sum'),
    Units   =('Quantity','sum'),
    Profit  =('Profit','sum'),
    Orders  =('OrderID','count')
).sort_values('Revenue', ascending=False)
prod['Margin_pct'] = (prod['Profit'] / prod['Revenue'] * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].barh(prod.index, prod['Revenue']/1000, color='steelblue')
axes[0].set_xlabel('Revenue ($K)'); axes[0].set_title('Revenue by Product', fontweight='bold')
axes[1].barh(prod.index, prod['Margin_pct'], color='coral')
axes[1].set_xlabel('Profit Margin (%)'); axes[1].set_title('Profit Margin by Product', fontweight='bold')
plt.tight_layout()
plt.savefig('visualizations/product_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: visualizations/product_performance.png")
print(prod[['Revenue','Margin_pct','Orders']].head(5))


In [ ]:
# ── RFM Customer Segmentation ─────────────────────────────────
current_date = df['OrderDate'].max()
rfm = df.groupby('CustomerID').agg(
    Recency  =('OrderDate', lambda x: (current_date - x.max()).days),
    Frequency=('OrderID', 'nunique'),
    Monetary =('SalesAmount', 'sum')
)
rfm['R'] = pd.qcut(rfm['Recency'].rank(method='first'),  4, labels=[4,3,2,1]).astype(int)
rfm['F'] = pd.qcut(rfm['Frequency'].rank(method='first'),4, labels=[1,2,3,4]).astype(int)
rfm['M'] = pd.qcut(rfm['Monetary'],                      4, labels=[1,2,3,4]).astype(int)
rfm['Score'] = rfm[['R','F','M']].sum(axis=1)

def segment(s):
    if s >= 11: return 'Champions'
    if s >= 9:  return 'Loyal'
    if s >= 7:  return 'Potential'
    if s >= 5:  return 'At Risk'
    return 'Lost'
rfm['Segment'] = rfm['Score'].apply(segment)

seg_cnt = rfm['Segment'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].pie(seg_cnt.values, labels=seg_cnt.index, autopct='%1.1f%%',
            colors=['#FFD700','#C0C0C0','#CD7F32','#FF6B6B','#808080'])
axes[0].set_title('Customer Segments', fontweight='bold')

seg_spend = rfm.groupby('Segment')['Monetary'].mean().sort_values()
axes[1].barh(seg_spend.index, seg_spend.values/1000, color='teal')
axes[1].set_xlabel('Avg Spend ($K)'); axes[1].set_title('Avg Spend by Segment', fontweight='bold')
plt.tight_layout()
plt.savefig('visualizations/customer_segmentation.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: visualizations/customer_segmentation.png")


In [ ]:
# ── Regional analysis ─────────────────────────────────────────
reg = df.groupby('Region').agg(
    Revenue =('SalesAmount','sum'),
    AvgOrder=('SalesAmount','mean'),
    Customers=('CustomerID','nunique')
).sort_values('Revenue', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].bar(reg.index, reg['Revenue']/1000, color='#3498db')
axes[0].set_ylabel('Revenue ($K)'); axes[0].set_title('Revenue by Region', fontweight='bold')
axes[1].bar(reg.index, reg['AvgOrder'], color='#e74c3c')
axes[1].set_ylabel('Avg Order ($)'); axes[1].set_title('Avg Order Value by Region', fontweight='bold')
plt.tight_layout()
plt.savefig('visualizations/regional_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: visualizations/regional_analysis.png")
print(reg)


In [ ]:
# ── Correlation heatmap ───────────────────────────────────────
num_cols = ['SalesAmount','Quantity','UnitPrice','DiscountPercent','CustomerAge','Profit']
utils.plot_correlation_matrix(df[num_cols],
                               save_path='visualizations/correlation_matrix.png')
print("✅ Saved: visualizations/correlation_matrix.png")


In [ ]:
# ── Executive Summary ─────────────────────────────────────────
total_rev  = df['SalesAmount'].sum()
total_prof = df['Profit'].sum()
margin     = total_prof / total_rev * 100
aov        = df['SalesAmount'].mean()
customers  = df['CustomerID'].nunique()

print(f"""
╔══════════════════════════════════════════╗
║   PROJECT 1 – EXECUTIVE SUMMARY          ║
╠══════════════════════════════════════════╣
║ Total Revenue   : ${total_rev:>12,.2f}   ║
║ Total Profit    : ${total_prof:>12,.2f}   ║
║ Profit Margin   : {margin:>11.1f}%   ║
║ Avg Order Value : ${aov:>12.2f}   ║
║ Unique Customers: {customers:>12,}   ║
╚══════════════════════════════════════════╝
""")
df.to_csv('processed_sales_data.csv', index=False)
rfm.to_csv('customer_rfm.csv')
print("✅ Project 1 complete — all PNGs and CSVs saved.")
